# Exploring the Truck Data
* extract from S3
* transform
* create visualisaitons and reccomendations for T3


## Imports

In [ ]:
import pandas as pd
import altair as alt

## Load and transform data into data frame

In [16]:
# Load the cleaned truck data
df = pd.read_csv('cleaned_truck_data.csv')

# Convert 'at' column to datetime format for time-based analysis
df['at'] = pd.to_datetime(df['at'])

# Display summary statistics for numerical columns
print("\nColumns:")
print(df.columns)


Columns:
Index(['payment_method_id', 'truck_id', 'transaction_id', 'total', 'at',
       'truck_name', 'truck_description', 'has_card_reader', 'fsa_rating',
       'payment_method'],
      dtype='str')


## Key Insights


In [41]:

print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)

# === Calculate total revenue by truck
revenue_by_truck = df.groupby('truck_name')[
    'total'].sum().sort_values(ascending=False).reset_index()
top_truck = revenue_by_truck.iloc[0]['truck_name']
top_truck_revenue = revenue_by_truck.iloc[0]['total']

print(f"\n===Top Performing Truck: {top_truck}")
print(f"===Revenue: ${top_truck_revenue:,.2f}")


# === Number of transactions by payment method (count)
transactions_count_by_payment = df['payment_method'].value_counts()
total_transactions = len(df)
print(f"\n===Transactions count by Payment Methods===")
for payment_method, count in transactions_count_by_payment.items():
    percentage = (count / total_transactions) * 100
    print(f"{payment_method}: {count} transactions ({percentage:.2f}%)")

    # Statistical analysis of transaction amounts by payment method
print(f"\n===Transaction Amount Statistics by Payment Method===")

for payment_method in df['payment_method'].unique():
    method_data = df[df['payment_method'] == payment_method]['total']
    
    print(f"\n{payment_method.upper()}:")
    print(f"  Min:    ${method_data.min():,.2f}")
    print(f"  Max:    ${method_data.max():,.2f}")
    print(f"  Mean:   ${method_data.mean():,.2f}")
    print(f"  Median: ${method_data.median():,.2f}")
    print(f"  Mode:   ${method_data.mode()[0]:,.2f}")

# === Calculate total revenue across all trucks
total_revenue = df['total'].sum()
print(f"\n===Total Revenue Across All Trucks: ${total_revenue:,.2f}")

print("\n" + "="*70)


KEY INSIGHTS

===Top Performing Truck: Kings of Kebabs
===Revenue: $739,796.00

===Transactions count by Payment Methods===
card: 2198 transactions (50.86%)
cash: 2124 transactions (49.14%)

===Transaction Amount Statistics by Payment Method===

CARD:
  Min:    $99.00
  Max:    $1,299.00
  Mean:   $654.96
  Median: $700.00
  Mode:   $700.00

CASH:
  Min:    $99.00
  Max:    $1,299.00
  Mean:   $641.17
  Median: $599.00
  Mode:   $500.00

===Total Revenue Across All Trucks: $2,801,433.00



#### Conclusions from these key insights
* Card transactions skew slightly higher in value
* Cash customers cluster around $500, card customers around $700
* Both have the same range ($99-$1,299)

Then we can reccomend things like:
* Incentivising card payments for higher-value items?
* Understanding why cash customers spend less on average?

## Visualisations

In [42]:
# ============================================================================
# VISUALIZATION 1: REVENUE BY TRUCK (Horizontal Bar Chart)
# ============================================================================

# Create horizontal bar chart showing revenue by truck
# Bars are sorted by revenue (highest to lowest)
revenue_by_truck_chart = alt.Chart(revenue_by_truck).mark_bar().encode(
    x=alt.X('total:Q', title='Total Revenue ($)'),
    y=alt.Y('truck_name:N', sort='-x', title='Truck Name'),
    tooltip=['truck_name:N', alt.Tooltip(
        'total:Q', format='$,', title='Revenue')]
    , color=alt.Color('truck_name:N', legend=None)
).properties(
    title='Total Revenue by Truck',
    width=600,
    height=300
)
revenue_by_truck_chart

alt.Chart(...)

In [84]:
# Calculate percentages first
transactions_count_by_payment['percentage'] = (
    transactions_count_by_payment['count'] /
    transactions_count_by_payment['count'].sum() * 100
)
transactions_count_by_payment

# Base chart
base = alt.Chart(transactions_count_by_payment).encode(
    theta=alt.Theta(field="count", type="quantitative"),
    color=alt.Color(field="payment_method", type="nominal"),
    tooltip=['payment_method:N', 'count:Q', alt.Tooltip(
        'percentage:Q', format='.1f', title='Percentage (%)')]  
)

# Just the pie, no text
revenue_by_payment = base.mark_arc().properties(
    title='Distribution by Payment Method Count',
    width=400,
    height=400
)
revenue_by_payment

alt.Chart(...)

In [51]:
# Prepare data
stats_data = pd.DataFrame({
    'Statistic': ['Min', 'Max', 'Mean', 'Median', 'Mode'],
    'CARD': [99, 1299, 654.96, 700, 700],
    'CASH': [99, 1299, 641.17, 599, 500]
})

# Reshape for Altair
stats_long = stats_data.melt(
    id_vars='Statistic', var_name='Payment Method', value_name='Amount')

# Create grouped bar chart
alt.Chart(stats_long).mark_bar().encode(
    x='Statistic:N',
    y='Amount:Q',
    color='Payment Method:N',
    xOffset='Payment Method:N',
    tooltip=['Payment Method', 'Statistic', 'Amount']
).properties(title='Transaction Statistics by Payment Method')

alt.Chart(...)

In [88]:

# 6. VISUALIZATION 4: DAILY SALES TREND (Line Chart with Points)
# ============================================================================


df['date'] = df['at'].dt.date.astype(str)

# Aggregate data: sum total revenue for each date
daily_sales_df = df.groupby('date')['total'].sum().reset_index()

# Create line chart with points showing daily sales trend over time
sales_over_time = alt.Chart(daily_sales_df).mark_line(point=True).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('total:Q', title='Total Revenue ($)'),
    tooltip=[
        alt.Tooltip('date:T', title='Date'),
        alt.Tooltip('total:Q', format='$,.2f', title='Revenue')
    ]
).properties(
    title='Daily Sales Trend',
    width=700,
    height=300
)
sales_over_time

alt.Chart(...)

# Key findings and reccomendations will be in the README.md file